# Препроцессинг price / deals

- Переименование колонок с русского на английские имена (snake_case).
- Составной ключ `unit_match_key` для сопоставления строк между прайсом и сделками: семь полей в фиксированном порядке, разделитель `@`.

Порядок полей в ключе: **project_id → building_id → section → conditional_number → floor → area → room_count**.

Для прайса: при **Студия** в `unit_typology` в ключе комнатность считается **1**, значение `room_count` в таблице не меняется (в данных часто 0).

In [11]:
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd if (_cwd / "data").is_dir() else _cwd.parent
DATA_DIR = REPO_ROOT / "data"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.unit_key import (
    UNIT_KEY_PARTS_EN,
    UNIT_KEY_PARTS_RU,
    add_unit_match_key,
    fix_price_room_count_from_pool,
)

## 1. Словари переименования (RU → EN)

Имена на английском отражают смысл поля; общие сущности в обеих таблицах получают **одинаковые** имена, чтобы ключ и дальнейшие джойны были согласованы.

In [12]:
PRICE_RENAME_MAP: dict[str, str] = {
    "pool": "pool",
    "ID Проекта": "project_id",
    "ID Проекта для окружения": "env_project_id",
    "ID Корпус": "building_id",
    "Проект": "project_name",
    "Девелопер": "developer",
    "Класс проекта": "project_class",
    "Регион": "region",
    "Округ": "macro_district",
    "Район": "district",
    "Старт продаж": "sales_start",
    "Сдача К": "completion_k",
    "Ключи": "keys_status",
    "Тип помещения": "premises_type",
    "Секция": "section",
    "Этаж": "floor",
    "Отделка в отчет": "finish_in_report",
    "Тип кв/ап": "unit_typology",
    "Комнатность": "room_count",
    "Площадь": "area",
    "Цена": "price",
    "Цена без скидки": "price_list",
    "Старый бюджет": "budget_old",
    "Бюджет без скидки": "budget_list",
    "Бюджет": "budget",
    "Скидка, %": "discount_pct",
    "Наличие бюджета": "has_budget",
    "Изменение цены последнее": "last_price_change",
    "Дата файла": "file_date",
    "Начало/конец месяца": "month_span",
    "Источник": "source",
    "Период": "period",
    "lat": "lat",
    "lng": "lng",
    "Условный номер": "conditional_number",
    "Отделка К": "finish_tier",
    "Отделка текст": "finish_text",
    "Договор К": "contract_type_k",
    "Стадия К": "stage_k",
    "Экспозиция": "exposure",
}

DEALS_RENAME_MAP: dict[str, str] = {
    "key": "deal_row_key",
    "lat": "lat",
    "lng": "lng",
    "ID Проекта": "project_id",
    "ID Проекта для окруженияSales": "env_project_id_sales",
    "ID Корпус": "building_id",
    "Проект": "project_name",
    "Девелопер": "developer",
    "Класс проекта": "project_class",
    "Регион": "region",
    "Округ": "macro_district",
    "Район": "district",
    "Округ Направление": "macro_district_direction",
    "Комнатность": "room_count",
    "Этаж": "floor",
    "Условный номер": "conditional_number",
    "Площадь": "area",
    "Тип помещения": "premises_type",
    "Наличие бюджета": "has_budget",
    "Цена": "price",
    "Цена без скидки": "price_list",
    "Бюджет": "budget",
    "Бюджет без скидки": "budget_list",
    "Ключи": "keys_status",
    "Корпус": "building_name",
    "Покупатель ФЛ": "buyer_person",
    "Покупатель ЮЛ": "buyer_company",
    "Номер регистрации": "registration_number",
    "Дата регистрации": "registration_date",
    "Дата регистрации_": "registration_date_raw",
    "Залогодержатель/Банк": "pledge_holder_bank",
    "Длительность обременения": "pledge_duration_months",
    "Тип обременения": "pledge_type",
    "Отделка": "finish",
    "Дата подписания": "signing_date",
    "Дата ДДУ": "ddu_date",
    "Срок сдачи": "completion_due",
    "Стадия строительства": "construction_stage",
    "Ипотека": "mortgage",
    "Секция": "section",
    "Старт продаж": "sales_start",
    "Продавец ЮЛ": "seller_company",
    "Продавец ФЛ ID": "seller_person_id",
    "Описание помещения": "premises_description",
    "Опт": "is_wholesale",
    "Стадия строительства в дату ДДУ": "construction_stage_at_ddu",
    "Цена ДДУ": "price_ddu",
    "Тип сделки": "deal_type",
    "Дата регистрации модель": "registration_date_model",
    "Цена кв. м": "price_per_sqm",
    "ID дом.рф": "dom_rf_id",
    "Бюджет по ПД": "budget_pd",
    "Цена по ПД": "price_pd",
}

## 2. Ключ для мэтча квартир (price ↔ deals)

Семь компонент в порядке: `ID Проекта`, `ID Корпус`, `Секция`, `Условный номер`, `Этаж`, `Площадь`, `Комнатность` — значения склеиваются через `@`. Нет данных (`NaN`, пусто, только пробелы, `-` и длинное тире) → пустой сегмент, подряд может быть `@@`.

**Площадь:** сначала округление до одного десятичного знака; если получилось целое — в ключ без `.0` (например `248`); иначе одна цифра после запятой с запятой как разделителем (`204,2`). Нечисловое/пустое — пустой сегмент.

**Секция (`section`):** значение в ключе не «нормализуем» как число: без округления; если в данных дробное, в ключе десятичный разделитель — **точка** (запятая в строке заменяется на точку). Произвольные текстовые коды секции не трогаем.

После переименования те же поля: `project_id`, `building_id`, `section`, `conditional_number`, `floor`, `area`, `room_count`.

Реализация ключа — в **`utils/unit_key.py`**: `add_unit_match_key`, `fix_price_room_count_from_pool`, константы `UNIT_KEY_PARTS_EN` / `UNIT_KEY_PARTS_RU`. Импорт выполняется в первой ячейке.


## 3. Пайплайн: переименование + ключ

Для **price**: если ``Комнатность`` / ``room_count`` не число (например ``пх``), в строке подставляется **последний сегмент** поля ``pool`` (после последнего ``@``), колонка ``room_count`` обновляется, затем строится ключ.

In [41]:
def rename_price_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Переименовывает известные RU-колонки; остальные (напр. `price_slice`) не трогает."""
    m = {c: PRICE_RENAME_MAP[c] for c in df.columns if c in PRICE_RENAME_MAP}
    return df.rename(columns=m)


def rename_deals_columns(df: pd.DataFrame) -> pd.DataFrame:
    m = {c: DEALS_RENAME_MAP[c] for c in df.columns if c in DEALS_RENAME_MAP}
    return df.rename(columns=m)


def preprocess_price(df: pd.DataFrame, *, key_col: str = "unit_match_key") -> pd.DataFrame:
    """Переименование прайса, правка нечисловой ``room_count`` из ``pool``, затем ``unit_match_key``."""
    x = rename_price_columns(df)
    x = fix_price_room_count_from_pool(x)
    return add_unit_match_key(
        x,
        renamed=True,
        out_col=key_col,
        studio_typology_col="unit_typology",
        studio_typology_value="Студия",
        studio_key_room_count=1,
    )


def preprocess_deals(df: pd.DataFrame, *, key_col: str = "unit_match_key") -> pd.DataFrame:
    """Переименование колонок сделок + unit_match_key."""
    return add_unit_match_key(rename_deals_columns(df), renamed=True, out_col=key_col)

## 4. Пример (небольшой семпл)

Прайс читается из **нескольких** `price_dataflat_*.csv` в `data/`; из каждого файла берётся по ``N`` строк (или весь файл, если ``N is None``). Колонка ``price_slice`` помечает срез.

In [46]:
PRICE_CSV_SLICES: dict[str, str] = {
    "2020_2025": "price_dataflat_2020_2025.csv",
    "2023_01_06": "price_dataflat_2023_01_06.csv",
    "2023_04_12": "price_dataflat_2023_04_12.csv",
    "2023_07_12": "price_dataflat_2023_07_12.csv",
    "2024_01_06": "price_dataflat_2024_01_06.csv",
    "2024_07_12": "price_dataflat_2024_07_12.csv",
    "2025_01_05": "price_dataflat_2025_01_05.csv",
}

N = 50_000  # строк из каждого price CSV; None = прочитать файлы целиком

price_parts = []
for slice_label, csv_name in PRICE_CSV_SLICES.items():
    path = DATA_DIR / csv_name
    if not path.is_file():
        print(f"skip (нет файла): {path}")
        continue
    read_kw = dict(low_memory=False)
    if N is not None:
        read_kw["nrows"] = N
    part = pd.read_csv(path, **read_kw)
    part = part.assign(price_slice=slice_label)
    price_parts.append(part)

if not price_parts:
    raise FileNotFoundError("Не найден ни один файл из PRICE_CSV_SLICES в DATA_DIR")

price_raw = pd.concat(price_parts, ignore_index=True)
deals_raw = pd.read_csv(DATA_DIR / "deals_dataflat_2020_2025.csv", nrows=N if N is not None else None, low_memory=False)

price_pp = preprocess_price(price_raw)
deals_pp = preprocess_deals(deals_raw)

print(price_pp[["unit_match_key", "project_id", "building_id", "section"]].head(3))
print(deals_pp[["unit_match_key", "project_id", "building_id", "section"]].head(3))

             unit_match_key  project_id  building_id section
0  534@1419@1@108@5@248,2@4         534         1419       1
1  536@1422@1@44@12@122,2@4         536         1422       1
2   536@1422@5@139@5@53,7@1         536         1422       5
             unit_match_key  project_id building_id section
0   551@8654@@428@48@38,5@1         551        8654       -
1   551@7450@@375@52@34,6@1         551        7450       -
2  551@7450@@210@31@107,9@4         551        7450       -


In [48]:
# Сравнение ``pool`` и ``unit_match_key`` + классификация расхождений
from collections import Counter

SEG_NAMES = list(UNIT_KEY_PARTS_EN)


def _split_segments(s):
    if pd.isna(s):
        return [""] * 7
    parts = str(s).strip().split("@")
    return (parts + [""] * 7)[:7]


def classify_pool_key_diff(pool_s, key_s):
    pa, ka = _split_segments(pool_s), _split_segments(key_s)
    diff_i = [i for i in range(7) if pa[i] != ka[i]]
    if not diff_i:
        return "OK", []
    parts = []
    for i in diff_i:
        pn, kn = pa[i], ka[i]
        if i == 2:
            if pn == "-" and kn == "":
                parts.append("section: «-» в pool → пустой сегмент в ключе")
            else:
                try:
                    if abs(float(pn.replace(",", ".")) - float(kn.replace(",", "."))) < 1e-9:
                        parts.append("section: одно число, разный текст (float-шум / формат)")
                    else:
                        parts.append(f"section: разные {pn!r} vs {kn!r}")
                except ValueError:
                    parts.append(f"section: {pn!r} vs {kn!r}")
        elif i == 3:
            try:
                fd = abs(float(pn.replace(",", ".")) - float(kn.replace(",", ".")))
                if fd < 1e-6:
                    parts.append("conditional_number: одно число, разный текст (нули / float)")
                else:
                    parts.append(f"conditional_number: {pn!r} vs {kn!r}")
            except ValueError:
                parts.append(f"conditional_number: {pn!r} vs {kn!r}")
        else:
            parts.append(f"{SEG_NAMES[i]}: {pn!r} vs {kn!r}")
    return " · ".join(parts), diff_i


mask = price_pp["pool"].astype(str) != price_pp["unit_match_key"].astype(str)
pool_key_mismatches = price_pp.loc[mask].copy()

n_tot, n_bad = len(price_pp), len(pool_key_mismatches)
print(
    f"Всего строк: {n_tot}, несовпадений pool vs unit_match_key: {n_bad} "
    f"({100 * n_bad / max(n_tot, 1):.3f}%)"
)

by_segment = Counter()
by_reason = Counter()
for _, row in pool_key_mismatches.iterrows():
    reason, diff_i = classify_pool_key_diff(row["pool"], row["unit_match_key"])
    by_reason[reason] += 1
    for j in diff_i:
        by_segment[SEG_NAMES[j]] += 1

print("\nЧастота по сегментам:")
for name, cnt in by_segment.most_common():
    print(f"  {name}: {cnt}")

print("\nТипы расхождений:")
for reason, cnt in by_reason.most_common(30):
    print(f"  {cnt:5d}  {reason}")

pool_key_mismatches

Всего строк: 350000, несовпадений pool vs unit_match_key: 18714 (5.347%)

Частота по сегментам:
  section: 14223
  area: 5000
  room_count: 2064
  conditional_number: 1982

Типы расхождений:
  10116  section: «-» в pool → пустой сегмент в ключе
   1270  room_count: '0' vs '1'
    769  conditional_number: '-' vs ''
    342  section: «-» в pool → пустой сегмент в ключе · room_count: '0' vs '1'
    221  section: «-» в pool → пустой сегмент в ключе · conditional_number: '-' vs ''
    184  conditional_number: одно число, разный текст (нули / float)
     96  room_count: '4' vs '4.0'
     88  section: одно число, разный текст (float-шум / формат)
     82  room_count: '2' vs '2.0'
     66  room_count: '0' vs '0.0'
     55  section: 'Позиция 203, секция В' vs 'Позиция 203. секция В'
     54  section: 'Позиция 202, секция В' vs 'Позиция 202. секция В'
     52  room_count: '3' vs '3.0'
     52  conditional_number: '-' vs '' · room_count: '0' vs '1'
     51  area: '22,81' vs '22,8'
     48  area: 

,pool,project_id,env_project_id,building_id,project_name,developer,project_class,region,macro_district,district,...,lat,lng,conditional_number,finish_tier,finish_text,contract_type_k,stage_k,exposure,price_slice,unit_match_key
178,555@1458@@@75@574@4,555,555,1458,Город Столиц,Capital Group,премиум,Москва,ЦАО,Пресненский,...,55.74723,37.53846,-,2 - Частично,-,ДКП,сдан_ГК,464,2020_2025,555@1458@@@75@574@4.0
234,604@1541@@@5@212@4,604,604,1541,Арт Хаус,State Development,де-люкс,Москва,ЦАО,Таганский,...,55.75007,37.64967,-,0 - Нет,-,ДКП,сдан_ГК,476,2020_2025,604@1541@@@5@212@4.0
235,604@1541@@@5@249@4,604,604,1541,Арт Хаус,State Development,де-люкс,Москва,ЦАО,Таганский,...,55.75007,37.64967,-,0 - Нет,-,ДКП,сдан_ГК,476,2020_2025,604@1541@@@5@249@4.0
536,"626@1590@1@609.20000000000005@6@82,8@0",626,626,1590,Звезды Арбата,БСК,де-люкс,Москва,ЦАО,Арбат,...,55.75331,37.58066,609.2,2 - Частично,Без отделки,ДКП,сдан_ГК,417,2020_2025,"626@1590@1@609.2@6@82,8@0"
633,"640@10027@2@61Л@13@258,6@4",640,640,10027,Садовые кварталы,Интеко,де-люкс,Москва,ЦАО,Хамовники,...,55.72639,37.57246,61Л,0 - Нет,-,ДКП,сдан_ГК,24,2020_2025,"640@10027@2@61Л@13@258,6@4.0"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349948,"5611@13406@2.2999999999999998@124@4@38,3@1",5611,5611,13406,Светлый мир БиоПолиc,Seven Suns Development,комфорт,Московская область,Ленинский городской округ,Мещерино (п.),...,55.51617,37.81048,124,0 - Нет,-,Эскроу,идёт отделка,1238,2025_01_05,"5611@13406@2.3@124@4@38,3@1"
349949,5611@13406@2.2999999999999998@125@4@38@1,5611,5611,13406,Светлый мир БиоПолиc,Seven Suns Development,комфорт,Московская область,Ленинский городской округ,Мещерино (п.),...,55.51617,37.81048,125,0 - Нет,-,Эскроу,идёт отделка,1238,2025_01_05,5611@13406@2.3@125@4@38@1
349950,"5611@13406@2.2999999999999998@128@4@57,4@2",5611,5611,13406,Светлый мир БиоПолиc,Seven Suns Development,комфорт,Московская область,Ленинский городской округ,Мещерино (п.),...,55.51617,37.81048,128,0 - Нет,-,Эскроу,идёт отделка,1238,2025_01_05,"5611@13406@2.3@128@4@57,4@2"
349951,"5611@13406@2.2999999999999998@129@4@57,4@2",5611,5611,13406,Светлый мир БиоПолиc,Seven Suns Development,комфорт,Московская область,Ленинский городской округ,Мещерино (п.),...,55.51617,37.81048,129,0 - Нет,-,Эскроу,идёт отделка,1238,2025_01_05,"5611@13406@2.3@129@4@57,4@2"


In [ ]:
price_pp.groupby(["room_count"]).value_counts()

room_count  pool                      project_id  env_project_id  building_id  project_name   developer          project_class  region  macro_district  district           sales_start  completion_k  keys_status  premises_type  section  floor  finish_in_report  unit_typology  area   price    price_list  budget_old  budget_list  budget     discount_pct  has_budget  last_price_change  file_date   month_span  source               period      lat       lng       conditional_number  finish_tier   finish_text  contract_type_k  stage_k  exposure  unit_match_key          
0           610@1552@@@2@136,6@0      610         610             1552         Меркурий-Сити  Rasen              де-люкс        Москва  ЦАО             Пресненский        01.03.2012   01.03.2013    -            Апартамент     -        2      Н/Д               -              136.6  729750   729750      99683850    99683850     99683850   0             Да          0                  01.01.2021  end         https://weknow.ru    01